# 18 — Files, paths and serialization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrshahabian/linguagent-labs/blob/main/units/python/18-files-and-serialization/lab.ipynb) [![Open in LinguAgent](https://img.shields.io/badge/Open%20in-LinguAgent-1f6f5c)](https://linguagent.com/courses/python/files-and-serialization?utm_source=github&utm_medium=labs)

*~45 minutes · 25 tests*

Every real program eventually reads a config, writes a checkpoint, or logs
to a file it must not corrupt if the process dies mid-write. This module is
the mechanics behind all three: `pathlib` instead of string-joined paths,
`json` for the data, and context managers for the part people get wrong —
making sure cleanup happens whether or not the code in between raised.

Work in `exercises.py`.

**How this notebook works:** run the first cell; then, section by section, read, write the code in the cell under it, and run the check below it. Stuck? `lab.solution("name")` shows the reference — try first, it's worth it.

In [ ]:
#@title ▶️ Run me first — fetches this lab and its checker
import os, sys, subprocess
HOME = "/content/linguagent-labs"
if "google.colab" in sys.modules:
    if not os.path.isdir(HOME):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/mrshahabian/linguagent-labs.git", HOME], check=True)
else:  # Jupyter, VS Code or a Codespace: this notebook already sits inside the repo
    HOME = os.path.abspath(os.path.join(os.getcwd(), *[".."] * 3)) if os.path.isdir("../../../labkit") else os.getcwd()
sys.path.insert(0, HOME)
import labkit
lab = labkit.open("python/files-and-serialization")

In [ ]:
# What the exercises below use — run this cell.
import contextlib

import json

import os

from pathlib import Path

## What you'll build

| Name | You'll implement | Shows up in real code as |
|---|---|---|
| `read_lines` | `read_lines` | Reading a small file into lines without manual newline stripping |
| `list_python_files` | `list_python_files` | Recursive file discovery — `Path.rglob` instead of `os.walk` |
| `ensure_dir` | `ensure_dir` | The idempotent "make sure this output directory exists" call |
| `save_json`, `load_json` | both | The config/checkpoint-metadata round trip |
| `merge_configs` | `merge_configs` | Layering a user config over defaults — the real "deep merge" |
| `SetEncoder`, `dumps_with_sets` | `SetEncoder.default`, `dumps_with_sets` | Extending `json` for a type it doesn't know natively |
| `AtomicWriter` | `__enter__`, `__exit__` | A file write a reader never sees half-finished |
| `temporary_cwd` | `temporary_cwd` | A `@contextmanager` that guarantees cleanup, not just setup |
| `chunked_lines` | `chunked_lines` | Streaming a file that doesn't fit comfortably in memory |

---

## 1. `pathlib` over string paths

```python
Path(path).read_text().splitlines()
Path(directory).rglob("*.py")
```

`pathlib.Path` replaces `os.path.join`, `os.walk`, and manual
`open()`/`.read()`/`.close()` with methods on an object that represents
the path itself. `rglob("*.py")` is the recursive glob — it walks every
subdirectory looking for matches, the same job `os.walk` plus a filename
filter does, in one call. `read_text()` opens, reads, and closes the file
for you; `.splitlines()` on the result gives lines with no trailing `\n`
to strip, unlike iterating an open file object.

```python
>>> read_lines("notes.txt")
['line1', 'line2', 'line3']
>>> list_python_files("units/python/18-files-and-serialization")
['units/python/18-files-and-serialization/exercises.py', ...]
```

**In `exercises.py`:** `read_lines`, `list_python_files`.

In [ ]:
def read_lines(path):
    """Return the file's lines as a list of strings, with no trailing newline.

    Path.read_text().splitlines() reads the whole file at once and never
    leaves a dangling '\\n' on each line the way iterating the open file
    object does - no need for `.rstrip("\\n")` on every line yourself.
    """
    raise NotImplementedError("your code here")

In [ ]:
def list_python_files(directory):
    """Return every .py file under directory, recursively, sorted by path.

    Path.rglob("*.py") walks subdirectories automatically - the
    recursive-glob equivalent of os.walk plus a filename filter, without
    the manual os.path.join bookkeeping. Sorting makes the result
    deterministic; filesystem iteration order is not guaranteed.
    """
    raise NotImplementedError("your code here")

In [ ]:
lab.check("§1", globals())

## 2. Idempotent directory creation

```python
Path(path).mkdir(parents=True, exist_ok=True)
```

`parents=True` is `mkdir -p` — it creates any missing intermediate
directories instead of raising `FileNotFoundError` when a parent is
missing. `exist_ok=True` is what makes calling this twice safe: without
it, a bare `.mkdir()` raises `FileExistsError` the second time, which is
almost never what "make sure this directory exists" actually wants.

```python
>>> ensure_dir("output/run-42/checkpoints")
PosixPath('output/run-42/checkpoints')
>>> ensure_dir("output/run-42/checkpoints")   # calling again is a no-op
PosixPath('output/run-42/checkpoints')
```

**In `exercises.py`:** `ensure_dir`.

In [ ]:
def ensure_dir(path):
    """Create path (and any missing parents) if it doesn't exist; return it as a Path.

    parents=True creates intermediate directories the way `mkdir -p`
    does. exist_ok=True makes this idempotent - calling it twice is not
    an error, unlike a bare Path.mkdir() which raises FileExistsError the
    second time.
    """
    raise NotImplementedError("your code here")

In [ ]:
lab.check("§2", globals())

## 3. JSON round trips and deep merges

```python
json.dumps(data, indent=2)
json.loads(text)
```

The straightforward half. The interesting exercise is `merge_configs`:

```python
def merge_configs(base, override):
    result = dict(base)
    for key, value in override.items():
        if (key in result
                and isinstance(result[key], dict)
                and isinstance(value, dict)):
            result[key] = merge_configs(result[key], value)
        else:
            result[key] = value
    return result
```

`{**base, **override}` only merges **one level deep** — if both have a
`"nested"` key that's itself a dict, the whole nested dict from `override`
replaces the one from `base`, instead of the two merging. That is almost
never what "layer a user config over the defaults" means in practice.
This walks every key that is a dict on **both** sides and recurses, so
only the leaf values that `override` actually specifies get replaced.

```python
>>> base = {"a": 1, "nested": {"x": 1, "y": 2}}
>>> override = {"nested": {"y": 99, "z": 3}, "b": 2}
>>> merge_configs(base, override)
{'a': 1, 'nested': {'x': 1, 'y': 99, 'z': 3}, 'b': 2}
```

**In `exercises.py`:** `save_json`, `load_json`, `merge_configs`.

In [ ]:
def save_json(data, path):
    """Write data to path as JSON, creating the file (or overwriting it)."""
    raise NotImplementedError("your code here")

In [ ]:
def load_json(path):
    """Read and parse the JSON file at path."""
    raise NotImplementedError("your code here")

In [ ]:
def merge_configs(base, override):
    """Recursively merge override into base; override wins on every leaf.

    A plain `{**base, **override}` only merges one level deep - a nested
    dict in override completely replaces the matching one in base instead
    of merging with it. This walks matching keys that are dicts on BOTH
    sides and merges those recursively too, which is the behaviour people
    actually mean by "merge these two config files."
    """
    raise NotImplementedError("your code here")

In [ ]:
lab.check("§3", globals())

## 4. Extending `json` for a type it doesn't know

```python
>>> json.dumps({"tags": {"a", "b"}})
Traceback (most recent call last):
    ...
TypeError: Object of type set is not JSON serializable
```

JSON has no set type, so `json.dumps` raises on one. `json.JSONEncoder`
has a documented extension point for exactly this: override `.default()`,
which is called only for objects the base encoder doesn't already know
how to handle — everything already-serialisable (`dict`, `list`, `str`,
numbers, `bool`, `None`) still goes through the normal fast path
untouched.

```python
class SetEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, set):
            return sorted(obj)
        return super().default(obj)
```

The `super().default(obj)` fallback matters: it is what produces the
correct `TypeError` for a *genuinely* unsupported type (say, a socket),
instead of `.default()` silently returning `None` for it.

```python
>>> dumps_with_sets({"tags": {"b", "a", "c"}})
'{"tags": ["a", "b", "c"]}'
```

**In `exercises.py`:** `SetEncoder.default`, `dumps_with_sets`.

In [ ]:
class SetEncoder(json.JSONEncoder):
    """A JSONEncoder that can serialise sets by turning them into sorted lists.

    json.dumps raises TypeError on a bare set - JSON has no set type.
    Overriding .default() is the documented extension point: it is only
    called for objects the base encoder does not already know how to
    handle, so dict/list/str/int/float/bool/None all still go through the
    normal fast path.
    """

    def default(self, obj):
        raise NotImplementedError("your code here")

In [ ]:
def dumps_with_sets(data):
    """json.dumps, but able to serialise sets via SetEncoder."""
    raise NotImplementedError("your code here")

In [ ]:
lab.check("§4", globals())

## 5. Atomic writes: a class-based context manager

```python
class AtomicWriter:
    def __enter__(self):
        self._file = open(self.tmp_path, "w")
        return self._file

    def __exit__(self, exc_type, exc_value, traceback):
        self._file.close()
        if exc_type is None:
            os.replace(self.tmp_path, self.path)
        else:
            self.tmp_path.unlink(missing_ok=True)
        return False
```

Writing straight to the target path leaves a window where a concurrent
reader (or a crash mid-write) sees a **partial file**. `AtomicWriter`
writes to a sibling `.tmp` file instead, and only promotes it with
`os.replace()` — an atomic rename on POSIX and modern Windows — once the
`with` block exits cleanly. `__exit__`'s three arguments tell you whether
an exception happened; returning `False` (or nothing) lets that exception
keep propagating instead of being swallowed, which matters here since a
failed write should still fail loudly, just without corrupting the
target.

```python
>>> with AtomicWriter("checkpoint.json") as f:
...     f.write('{"step": 1000}')
>>> # if the block above raised instead, checkpoint.json is untouched
```

**In `exercises.py`:** `AtomicWriter.__enter__`, `AtomicWriter.__exit__`.

In [ ]:
class AtomicWriter:
    """Write a file atomically: readers never see a partially-written file.

    Writes go to a sibling `<name>.tmp` file first. On a clean exit,
    os.replace() atomically swaps it into place - on POSIX and modern
    Windows this rename is a single filesystem operation, so a concurrent
    reader sees either the old complete file or the new complete file,
    never a half-written one. On an exception, the temp file is removed
    instead of being promoted, so the target path is left untouched.
    """

    def __init__(self, path):
        self.path = Path(path)
        self.tmp_path = self.path.with_suffix(self.path.suffix + ".tmp")
        self._file = None

    def __enter__(self):
        raise NotImplementedError("your code here")

    def __exit__(self, exc_type, exc_value, traceback):
        raise NotImplementedError("your code here")

In [ ]:
lab.check("§5", globals())

## 6. `@contextmanager`: cleanup that survives an exception

```python
@contextlib.contextmanager
def temporary_cwd(path):
    previous = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous)
```

Same idea as `AtomicWriter`, generator-decorator style instead of a
class: code before `yield` is setup (`__enter__`), code after is teardown
(`__exit__`). The `finally` is what makes the restore unconditional — if
the code inside the `with` block raises, `os.chdir(previous)` still runs
before the exception propagates out. Skipping the `try/finally` and just
writing `os.chdir(previous)` after the `yield` would leave the process in
the wrong directory forever the first time a caller's code inside the
block raises.

```python
>>> with temporary_cwd("/tmp"):
...     os.getcwd()
'/tmp'
>>> os.getcwd()   # back to normal after the block
'/original/dir'
```

**In `exercises.py`:** `temporary_cwd`.

In [ ]:
@contextlib.contextmanager
def temporary_cwd(path):
    """Change the working directory for the duration of the `with` block.

    The restore lives in `finally`, so the original directory comes back
    even if the block raises - a context manager's whole job is exactly
    this pairing of setup and guaranteed teardown, generalised from
    open()/close() to any resource, here the process's cwd.
    """
    raise NotImplementedError("your code here")

In [ ]:
lab.check("§6", globals())

## 7. Streaming a file that doesn't fit in memory

```python
def chunked_lines(path, chunk_size):
    chunk = []
    with open(path) as f:
        for line in f:
            chunk.append(line.rstrip("\n"))
            if len(chunk) == chunk_size:
                yield chunk
                chunk = []
    if chunk:
        yield chunk
```

`path.read_text().splitlines()` — the approach from §1 — loads the
**entire file** before you can do anything with it. Fine for a config,
wrong for a log file too large to fit in memory. Iterating the open file
object one line at a time keeps memory proportional to `chunk_size`, not
to the file's size — the same shape as the batching exercise back in
module 15, applied to disk instead of a queue.

```python
>>> list(chunked_lines("access.log", chunk_size=3))
[['line1', 'line2', 'line3'], ['line4', 'line5', 'line6'], ['line7']]
```

**In `exercises.py`:** `chunked_lines`.

In [ ]:
def chunked_lines(path, chunk_size):
    """Yield the file's lines in lists of at most chunk_size, streaming.

    Reads line by line rather than calling read_text() and slicing the
    result, so memory use stays proportional to chunk_size instead of the
    whole file - the difference between working on a log file that fits
    in memory and one that does not.
    """
    raise NotImplementedError("your code here")

In [ ]:
lab.check("§7", globals())

---

## The exercises

| Function / class | What it teaches |
|---|---|
| `read_lines`, `list_python_files` | `pathlib` basics: `read_text`, `rglob` |
| `ensure_dir` | `mkdir(parents=True, exist_ok=True)` |
| `save_json`, `load_json` | The JSON round trip |
| `merge_configs` | A real deep merge vs. `{**a, **b}`'s one-level merge |
| `SetEncoder`, `dumps_with_sets` | Extending `json.JSONEncoder.default` |
| `AtomicWriter` | A class-based context manager; atomic rename |
| `temporary_cwd` | `@contextmanager`; `finally` guarantees teardown |
| `chunked_lines` | Streaming reads with bounded memory |

Every function in this table has a stub in `exercises.py` and test
coverage in `test_exercises.py` — a clean 1:1 mapping.

## Check your understanding

1. **Why does `{**base, **override}` fall short as a config-merging
   strategy?**
   <details><summary>Answer</summary>It only merges one level deep. If
   both dicts have a key whose value is itself a dict, the nested dict
   from <code>override</code> completely replaces the one from
   <code>base</code> rather than merging with it — any keys only present
   in <code>base</code>'s nested dict are silently lost.</details>

2. **What does `os.replace()` guarantee that writing directly to the
   target path does not?**
   <details><summary>Answer</summary>On POSIX (and modern Windows), a
   rename onto an existing path is a single atomic filesystem operation
   — any concurrent reader sees either the old complete file or the new
   complete file, never something half-written. Writing directly to the
   target leaves a window where a crash or a concurrent read catches the
   file mid-write.</details>

3. **Why does `AtomicWriter.__exit__` check `exc_type` before deciding
   whether to promote the temp file?**
   <details><summary>Answer</summary>If an exception happened inside the
   <code>with</code> block, the data written so far is presumed
   incomplete or invalid, so the temp file is discarded instead of being
   renamed into place — the target path is left exactly as it was
   before the write was attempted.</details>

4. **In a `@contextmanager` generator, what's the difference between
   putting the teardown code after `yield` versus after `yield` inside a
   `finally` block?**
   <details><summary>Answer</summary>Without <code>finally</code>, the
   teardown only runs if the code inside the <code>with</code> block
   completes without raising — an exception there skips straight past
   it, leaving the resource (a changed directory, an open connection,
   held lock) never cleaned up. <code>finally</code> makes the teardown
   run unconditionally, exception or not.</details>

5. **Why does `chunked_lines` iterate the open file object instead of
   calling `.read_text().splitlines()` and slicing the result?**
   <details><summary>Answer</summary><code>read_text()</code> loads the
   entire file into memory before any chunk can be produced, so peak
   memory use scales with the whole file's size. Iterating the file
   object line by line only ever holds one chunk's worth of lines at a
   time, which is the difference between working and OOMing on a file
   too large to fit in memory.</details>

## Interview questions this module answers

- *How would you merge a user config over a defaults dict, one level
  deep vs. properly nested?*
- *Why is `os.replace()` safer for writing output files than writing
  directly to the target path?*
- *What does `finally` guarantee in a context manager that comes after
  the `yield` — and what happens without it?*
- *This log file is bigger than memory — how do you process it?*
- *How do you make `json.dumps` handle a type it doesn't support
  natively?*

## Finished? Check everything, then record it

The first cell checks the whole unit. The second records your result in LinguAgent (needs your lab token from [https://linguagent.com/settings#labs](https://linguagent.com/settings#labs) — in Colab, add it once as a secret named `LINGUAGENT_TOKEN` under 🔑 in the left bar).

In [ ]:
lab.check_all(globals())

In [ ]:
lab.sync()

**Next:** [Python 10 - Testing and debugging patterns](https://colab.research.google.com/github/mrshahabian/linguagent-labs/blob/main/units/python/19-testing-and-debugging/lab.ipynb) · [the lesson in LinguAgent](https://linguagent.com/courses/python/files-and-serialization)